<a href="https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/w03_data_contract_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 3 — Structured Content Archetype Clustering** (locked in `w02_ml_task_framing`, superseding the Lane 2 draft from `w01_research_question`). Proxy target: `cluster_id`, discovered from the feature frame below — not an observed label.

**Schema confirmed live from the Hugging Face Dataset Viewer** (not guessed) for `fact_content_daily_performance`, `dim_content`, and `dim_clients` — see Section 1 for the exact column names this notebook uses.

In [3]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get("HF_TOKEN")  # Colab Secrets panel (key icon) -- never paste a token in a cell, this repo is public
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "month=2026-03"   # mid-panel month -- NOT the _sample table, which is the sealed final month (2026-06)
DAILY = f"{BASE}/fact_content_daily_performance/{MONTH}/*.parquet"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DIM_CLIENTS = f"{BASE}/dim_clients.parquet"

# Sanity check: confirm these columns actually exist in this parquet build before querying them.
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DAILY}') LIMIT 0").df()["column_name"].tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one pseudonymized content item**, identified by `(client_hash_id, content_hash_id)`, summarizing its search and engagement performance across **March 2026** (`2026-03-01` – `2026-03-31`).

`fact_content_daily_performance`'s native grain is `report_date × client_hash_id × content_hash_id` — one row per page per day, confirmed live in the HF Dataset Viewer: `report_date`, `client_hash_id`, `content_hash_id`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`, `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic/direct/referral/social/paid/ai`, `ai_chatgpt/perplexity/gemini/copilot/claude/meta/other`, `scroll_events`.

I aggregate that up to **page-month** with `GROUP BY client_hash_id, content_hash_id`, then join `dim_content` for static metadata. I'm using page-month, not page-day, because Lane 3 profiles what *kind* of page this is — an archetype is a monthly-stable property, not something that should flip because of one noisy Tuesday.

I chose `month=2026-03` specifically because it's a **mid-panel month**: the warehouse's `_sample` table is the final month (June 2026) and is reserved as a sealed test window, per the `flyrank-data` skill — developing feature logic there would mean developing inside my own future test set.

In [4]:
# Verify the daily grain claim: report_date x client_hash_id x content_hash_id should be unique.
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{DAILY}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate (report_date, client_hash_id, content_hash_id) combos found:", len(grain_probe))
grain_probe


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client_hash_id, content_hash_id) combos found: 0


,report_date,client_hash_id,content_hash_id,n


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature (daily fact, safe)** | monthly `SUM(gsc_impressions)`, `SUM(gsc_clicks)`, computed `ctr_month`, a **click-weighted** `avg_position_month` (`SUM(gsc_sum_position)/SUM(gsc_impressions)`, not a naive average of daily averages), `days_with_impressions`, `SUM(ga4_sessions)`, `engagement_rate_month` (`SUM(ga4_engaged_sessions)/SUM(ga4_sessions)`, only where `ga4_data_available IS TRUE`), `ai_session_share_month` (`SUM(sessions_ai)/SUM(ga4_sessions)`) | Every one is a sum or ratio built purely from March's own daily rows — nothing reaches past the window it describes. |
| **Feature (dim_content, use with caution)** | `word_count`, `content_type`, `main_intent`, `search_volume`, `competition_level` | Knowable *in principle*, but see the caveat below — `dim_content` is a **current snapshot** (export date 2026-07-03), not a point-in-time record as of March. |
| **Label / proxy** | `cluster_id` | Doesn't exist in the raw data — it's assigned *by* the clustering algorithm run on the features above. By definition it can never be a feature. |
| **Context** | `client_hash_id`, `content_hash_id` | Pseudonymous join/grouping keys only, per the `flyrank-data` skill's pseudonym rule — never fed to the model as a value it could learn from. |
| **Excluded** | raw daily rows (collapsed by design); `keyword_hash_id`/`url_hash_id`/`provider_used`/`model_used` (production metadata, not performance signal); `is_deleted` (filter criterion, not a feature — deleted content shouldn't be in the frame at all); `content_updated_date`, `last_optimized_date`, `optimization_eligible_date` (excluded from features for the **same snapshot reason** as below — these describe the page's state as of the July export, not as of March); any *within-month trend* column — reserved **only** for the illustrative leakage demo in Section 3, constructed the same way `trend_direction`/`trend_pct` are in the starter CSV. |

**The `dim_content` caveat, stated plainly:** rows in the live viewer show `content_updated_date` values of `2026-06-01`, `2026-07-01`, even `2026-07-06` — all *after* my March window. `dim_content` reflects each page's state **as of the export date**, not its state during March. Using `word_count`/`content_type` as March features assumes they didn't change between March and July — true for most static metadata, but not guaranteed, and not something this table can prove either way. I'm using them anyway (flagged, not silently), because Lane 3's archetypes need content metadata and the daily fact alone can't supply it — but this is a real, named limit on the contract, not an oversight.

In [5]:
# Section 2 has no new query -- it is a classification exercise over the schema confirmed above.
safe_daily_features = ["gsc_impressions_month", "gsc_clicks_month", "ctr_month",
                        "avg_position_month", "days_with_impressions",
                        "ga4_sessions_month", "engagement_rate_month", "ai_session_share_month"]
caution_snapshot_features = ["word_count", "content_type", "main_intent",
                              "search_volume", "competition_level"]
context_fields = ["client_hash_id", "content_hash_id"]
excluded_fields = ["keyword_hash_id", "url_hash_id", "provider_used", "model_used", "is_deleted",
                    "content_updated_date (snapshot risk)", "last_optimized_date (snapshot risk)",
                    "within_month_trend (demo-only, Section 3)"]

print("FEATURE (safe, daily fact):", safe_daily_features)
print("FEATURE (caution, snapshot):", caution_snapshot_features)
print("CONTEXT:", context_fields)
print("EXCLUDED:", excluded_fields)
print("LABEL/PROXY: cluster_id (assigned later, not a raw column)")


FEATURE (safe, daily fact): ['gsc_impressions_month', 'gsc_clicks_month', 'ctr_month', 'avg_position_month', 'days_with_impressions', 'ga4_sessions_month', 'engagement_rate_month', 'ai_session_share_month']
FEATURE (caution, snapshot): ['word_count', 'content_type', 'main_intent', 'search_volume', 'competition_level']
CONTEXT: ['client_hash_id', 'content_hash_id']
EXCLUDED: ['keyword_hash_id', 'url_hash_id', 'provider_used', 'model_used', 'is_deleted', 'content_updated_date (snapshot risk)', 'last_optimized_date (snapshot risk)', 'within_month_trend (demo-only, Section 3)']
LABEL/PROXY: cluster_id (assigned later, not a raw column)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three required checks — grain (done in Section 1), row count + date span, and availability with `IS TRUE` — then the five-feature frame, then the deliberate-leak trap.

In [6]:
# Query 2 of 3: slice row count + date span for the mid-panel month.
span = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT (client_hash_id, content_hash_id)) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM read_parquet('{DAILY}')
""").df()
print(span)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count  n_clients  n_content_items   min_date   max_date
0    9841378         55           331437 2026-03-01 2026-03-31


In [7]:
# Query 3 of 3: availability -- GA4 columns are only trustworthy where ga4_data_available IS TRUE.
# Using "= TRUE" or "NOT ga4_data_available" silently mishandles NULL rows -- IS TRUE / IS NOT TRUE
# is the only safe filter (same pattern applies to gsc_data_available).
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_unavailable_or_null_rows
    FROM read_parquet('{DAILY}')
""").df()
availability["pct_survive_ga4_filter"] = (
    availability["ga4_available_rows"] / availability["total_rows"] * 100
).round(1)
print(availability)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  ga4_unavailable_or_null_rows  \
0     9841378            413966.0                     9427412.0   

   pct_survive_ga4_filter  
0                     4.2  


### Five features (max), each with an "available when" line

In [8]:
features_df = con.sql(f"""
    WITH monthly AS (
        SELECT
            d.client_hash_id,
            d.content_hash_id,
            SUM(d.gsc_impressions)                                       AS gsc_impressions_month,
            SUM(d.gsc_clicks)                                            AS gsc_clicks_month,
            SUM(d.gsc_sum_position)                                      AS gsc_sum_position_month,
            COUNT(DISTINCT d.report_date) FILTER (WHERE d.gsc_impressions > 0) AS days_with_impressions,
            SUM(d.ga4_sessions)                                          AS ga4_sessions_month,
            SUM(d.ga4_engaged_sessions) FILTER (WHERE d.ga4_data_available IS TRUE) AS ga4_engaged_month,
            SUM(d.ga4_sessions) FILTER (WHERE d.ga4_data_available IS TRUE)         AS ga4_sessions_avail_month,
            SUM(d.sessions_ai)                                           AS sessions_ai_month
        FROM read_parquet('{DAILY}') d
        GROUP BY d.client_hash_id, d.content_hash_id
    )
    SELECT
        m.*,
        CASE WHEN m.gsc_impressions_month > 0
             THEN ROUND(100.0 * m.gsc_clicks_month / m.gsc_impressions_month, 2)
             ELSE NULL END AS ctr_month,
        CASE WHEN m.gsc_impressions_month > 0
             THEN ROUND(m.gsc_sum_position_month::DOUBLE / m.gsc_impressions_month, 2)
             ELSE NULL END AS avg_position_month,
        CASE WHEN m.ga4_sessions_avail_month > 0
             THEN ROUND(100.0 * m.ga4_engaged_month / m.ga4_sessions_avail_month, 2)
             ELSE NULL END AS engagement_rate_month,
        c.word_count
    FROM monthly m
    LEFT JOIN read_parquet('{DIM_CONTENT}') c
           ON c.client_hash_id = m.client_hash_id AND c.content_hash_id = m.content_hash_id
""").df()

print("feature frame shape:", features_df.shape)
features_df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame shape: (331437, 14)


,client_hash_id,content_hash_id,gsc_impressions_month,gsc_clicks_month,gsc_sum_position_month,days_with_impressions,ga4_sessions_month,ga4_engaged_month,ga4_sessions_avail_month,sessions_ai_month,ctr_month,avg_position_month,engagement_rate_month,word_count
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,936.0,29,NaN,NaN,NaN,NaN,0.00,5.17,NaN,2999
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,209.0,16,NaN,NaN,NaN,NaN,2.17,4.54,NaN,3057
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5237.0,31,NaN,NaN,NaN,NaN,0.11,5.83,NaN,2855
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,202.0,17,NaN,NaN,NaN,NaN,0.00,5.94,NaN,3281
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,21612.0,30,NaN,NaN,NaN,NaN,0.00,6.95,NaN,2779


1. **`gsc_impressions_month`** — knowable at the decision moment because it's a straight `SUM` of March's own `gsc_impressions`; nothing from April or later touches it.
2. **`avg_position_month`** — knowable because it's a **click-weighted** March average (`SUM(gsc_sum_position)/SUM(gsc_impressions)`), computed entirely from March rows — not a naive average-of-averages, which would over-weight low-volume days.
3. **`ctr_month`** — knowable because it's a derived ratio of two in-window sums (`gsc_clicks_month`, `gsc_impressions_month`), not a new source of future information.
4. **`engagement_rate_month`** — knowable *only* on the subset where `ga4_data_available IS TRUE`; outside that subset, GA4 wasn't collecting for this client yet, so the value is genuinely unknown, not zero — filtering, not imputing, keeps it honest.
5. **`word_count`** — knowable **with the caveat flagged in Section 2**: `dim_content` is a July-export snapshot, so this assumes the page's word count in March matched its word count at export time. Reasonable for most pages (word count rarely shrinks), but not provably true from this table alone — the honest "available when" answer here is *"knowable as of the export date, assumed stable back to March."*

### The trap: one label-derived column, on purpose

Lane 3 has no observed label to leak — `cluster_id` is discovered, not predicted. So to perform the leakage lesson from notebook 02 **on real warehouse data**, I build the same illustrative proxy the starter CSV uses: a `declining` flag, computed the same way as `trend_direction` — comparing the back half of March to the front half. This is a diagnostic side-quest to prove I can catch a leak, not a change of lane.

In [9]:
# Build an illustrative "declining" proxy the same way trend_direction is built in the starter
# CSV: last ~15 days of March vs first ~15 days, on the SAME daily table used for real features.
proxy_df = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impr_second_half
    FROM read_parquet('{DAILY}')
    GROUP BY client_hash_id, content_hash_id
    HAVING impr_first_half > 0
""").df()

proxy_df["trend_pct_demo"] = (
    (proxy_df["impr_second_half"] - proxy_df["impr_first_half"]) / proxy_df["impr_first_half"] * 100
)
proxy_df["declining_demo"] = (proxy_df["trend_pct_demo"] < -20).astype(int)

demo = features_df.merge(
    proxy_df[["client_hash_id", "content_hash_id", "trend_pct_demo", "declining_demo"]],
    on=["client_hash_id", "content_hash_id"], how="inner"
).dropna(subset=["gsc_impressions_month", "avg_position_month", "ctr_month", "declining_demo"])
print(demo.shape, "| decline rate:", demo["declining_demo"].mean().round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(151981, 16) | decline rate: 0.327


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HONEST_FEATURES = ["gsc_impressions_month", "avg_position_month", "ctr_month"]
X = demo[HONEST_FEATURES].fillna(0)
y = demo["declining_demo"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
print(f"HONEST quick score (AUC), safe features only: {honest_auc:.3f}")

# Now the trap: add trend_pct_demo itself as a "feature" -- it is LITERALLY the quantity the
# label is thresholded from (declining_demo = trend_pct_demo < -20). This is exactly the
# trend_direction/trend_pct mistake the flyrank-data skill calls out for the starter CSV.
LEAKY_FEATURES = HONEST_FEATURES + ["trend_pct_demo"]
Xl = demo[LEAKY_FEATURES].fillna(0)
Xltr, Xlte, yltr, ylte = train_test_split(Xl, y, test_size=0.3, random_state=42, stratify=y)
clf_leak = LogisticRegression(max_iter=1000).fit(Xltr, yltr)
leaky_auc = roc_auc_score(ylte, clf_leak.predict_proba(Xlte)[:, 1])
print(f"LEAKY quick score (AUC), with trend_pct_demo included: {leaky_auc:.3f}")
print(f"Jump: {honest_auc:.3f} -> {leaky_auc:.3f}")


HONEST quick score (AUC), safe features only: 0.609
LEAKY quick score (AUC), with trend_pct_demo included: 1.000
Jump: 0.609 -> 1.000


**Result:** the honest score sits well below 1.0 using only genuinely-March-knowable features. Adding `trend_pct_demo` — the exact quantity `declining_demo` is thresholded from — pushes AUC toward a near-perfect score, because the model isn't predicting anything anymore; it's reading the answer key. `trend_pct_demo` is deleted below and the honest number is what I keep.

In [11]:
# Delete the leaky column and keep the honest number.
del demo["trend_pct_demo"]
assert "trend_pct_demo" not in demo.columns
print("Kept, honest quick score (AUC):", round(honest_auc, 3))
print("Leaky column removed from the working frame.")


Kept, honest quick score (AUC): 0.609
Leaky column removed from the working frame.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel.** Not every client has GSC history reaching back to (or through) March 2026 — `dim_clients.gsc_data_start` differs per client. A page from a client whose history starts mid-March will show a partial, lower `days_with_impressions` for reasons that have nothing to do with its actual archetype. Checked below.
- **GA4 availability is three-valued, not two.** `ga4_data_available` can be `TRUE`, `FALSE`, or `NULL` — `NULL` means genuinely unknown, not "no engagement." `engagement_rate_month` is therefore missing-not-at-random: it's systematically absent for newer or smaller clients, not randomly scattered across the dataset.
- **Named limitation:** `dim_content` is a single frozen snapshot taken at export time (2026-07-03), not a time-series of content state. Fields like `word_count`, `content_type`, and `search_volume` describe the page **as it existed in July**, and this table has no way to tell me whether that matches how the page looked in March — `content_updated_date` values well past March (`2026-06-01`, `2026-07-01`, `2026-07-06`, observed directly in the live viewer) confirm real edits happened in between for at least some pages. Any archetype built using `word_count` is quietly assuming static-content stability that this data cannot verify.

In [12]:
# Verify the unbalanced-panel claim: how many clients had GSC history starting AFTER March began?
panel_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        SUM(CASE WHEN gsc_data_start > DATE '2026-03-01' THEN 1 ELSE 0 END) AS started_mid_march_or_later,
        SUM(CASE WHEN ga4_data_start IS NULL THEN 1 ELSE 0 END) AS null_ga4_start
    FROM read_parquet('{DIM_CLIENTS}')
""").df()
print(panel_check)

# Verify the dim_content snapshot claim directly: how many content items were updated AFTER March?
snapshot_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_content_items,
        SUM(CASE WHEN content_updated_date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS updated_after_march
    FROM read_parquet('{DIM_CONTENT}')
""").df()
snapshot_check["pct_updated_after_march"] = (
    snapshot_check["updated_after_march"] / snapshot_check["total_content_items"] * 100
).round(1)
print(snapshot_check)


   total_clients  started_mid_march_or_later  null_ga4_start
0            104                        15.0            53.0
   total_content_items  updated_after_march  pct_updated_after_march
0               519606             382739.0                     73.7
